In [1]:
import pandas as pd
import numpy as np

import psix
# check which version of psix 
print(psix.__version__)

import os 
import sys
import anndata as ad

# Import custom modules
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/beta-dirichlet-factor')
import factor_model
import waypoints_prep_input as wayp

from scipy.sparse import coo_matrix, csr_matrix
from sklearn.decomposition import PCA

0.11.1
Torch Version: 2.3.0+cu121
CUDA Version: 12.1


In [2]:
# Load the splice_adata simulated object 
sim_dir = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/Simulations/2025/manuscript_sim_analysis/AnndatasForSimulation"

# List the directories in the main path
dirs = os.listdir(sim_dir)

# Which dirs have "NoCellType_Random" in them 
dirs = [d for d in dirs if "NoCellType_Random" in d]
test_dir = dirs[0]

# load the adata object in the directory
adata = ad.read_h5ad(os.path.join(sim_dir, test_dir, "adata_input.h5ad"))

In [3]:
# Ensure 'start' and 'end' are numeric
adata.var["start"] = pd.to_numeric(adata.var["start"], errors="coerce")
adata.var["end"] = pd.to_numeric(adata.var["end"], errors="coerce")

# Extract junction metadata
junction_metadata = adata.var[["Cluster", "junction_id_index", "chr", "start", "end"]].copy()

# Extract read counts
junction_counts = pd.DataFrame(adata.layers["cell_by_junction_matrix"].toarray(), 
                               index=adata.obs_names, 
                               columns=adata.var_names)

# Group junctions by ATSE (Cluster)
grouped = junction_metadata.groupby("Cluster")

# Initialize storage for ATSE-level PSI and counts
atse_psi = pd.DataFrame(index=adata.obs_names)
atse_counts = pd.DataFrame(index=adata.obs_names)

for cluster, group in grouped:
    # Ensure exactly three junctions per ATSE
    if len(group) != 3:
        continue  

    # Identify J3 (Exon-skipping junction) → The longest spanning junction (min start, max end)
    j3_row = group[(group["start"] == group["start"].min()) & (group["end"] == group["end"].max())]

    # Ensure we found exactly one J3
    if j3_row.shape[0] != 1:
        continue  

    j3_index = j3_row["junction_id_index"].values[0]

    # Identify J1 & J2 (Exon-inclusion junctions) → Remaining two junctions
    j1_j2_rows = group.drop(j3_row.index)
    j1_index, j2_index = j1_j2_rows["junction_id_index"].values

    # Extract read counts
    j1_counts = junction_counts.loc[:, str(j1_index)]
    j2_counts = junction_counts.loc[:, str(j2_index)]
    j3_counts = junction_counts.loc[:, str(j3_index)]
    
    # Compute PSI for exon inclusion
    psi_values = (j1_counts + j2_counts) / (j1_counts + j2_counts + j3_counts)

    # Compute total ATSE read counts
    total_counts = j1_counts + j2_counts + j3_counts

    # Store results
    atse_psi[cluster] = psi_values
    atse_counts[cluster] = total_counts

In [4]:
# Replace NA with 0 in atse_psi and atse_counts
atse_psi.fillna(0, inplace=True)
atse_counts.fillna(0, inplace=True)

# Save PSI and counts DataFrames to tab-separated files
atse_psi_path = "atse_psi_matrix.tab.gz"
atse_counts_path = "atse_counts_matrix.tab.gz"

# rename columns as "ATSE_{}"
atse_counts.columns = ["ATSE_" + str(c) for c in atse_counts.columns] # cells by exons 
atse_psi.columns = ["ATSE_" + str(c) for c in atse_psi.columns] # cells by exons

# remove index name 
atse_psi.index.name = None

In [5]:
print("Percentage of zeros:", (atse_counts.T == 0).sum().sum() / atse_counts.T.size * 100, "%") 
print("Cells with all zero counts:", (atse_counts.T.sum(axis=1) == 0).sum())
print("Exons with all zero counts:", (atse_counts.T.sum(axis=0) == 0).sum())

# Identify exons (columns) with all zeros
valid_cells = atse_counts.T.columns[atse_counts.T.sum(axis=0) > 2000]
valid_exons = atse_counts.T.index[atse_counts.T.sum(axis=1) > 2000]

print("Valid cells:", len(valid_cells))
print("Valid exons:", len(valid_exons))

Percentage of zeros: 94.62448411760845 %
Cells with all zero counts: 0
Exons with all zero counts: 49
Valid cells: 18269
Valid exons: 4871


In [6]:
atse_counts.shape # cells by exons
# Subset to valid cells and exons
atse_counts = atse_counts.loc[valid_cells, valid_exons]
atse_psi = atse_psi.loc[valid_cells, valid_exons]

# print new shapes 
atse_counts.shape # cells by exons
atse_psi.shape # cells by exons

(18269, 4871)

In [7]:
# Now do the same but on atse_psi 
print("PSI shape:", atse_psi.shape) # exons by cells now
zero_cells = (atse_psi.sum(axis=0) == 0)
# remove cells with all zero PSI values
atse_psi = atse_psi.loc[:, ~zero_cells]

# Find any junctions that have all zero PSI values 
zero_junctions = (atse_psi.sum(axis=1) == 0)
atse_psi = atse_psi.loc[~zero_junctions, :]

# remove zero_cells from atse_counts also
atse_counts = atse_counts.loc[:, ~zero_cells]
atse_counts = atse_counts.loc[~zero_junctions, :]

# print final dimensions of atse_counts and atse_psi
print("Final counts shape:", atse_counts.shape)
print("Final PSI shape:", atse_psi.shape)

PSI shape: (18269, 4871)
Final counts shape: (18269, 4855)
Final PSI shape: (18269, 4855)


In [8]:
# now check if atse_counts has any row or column with all zeros
print("Cells with all zero counts:", (atse_counts.sum(axis=1) == 0).sum())
print("Exons with all zero counts:", (atse_counts.sum(axis=0) == 0).sum())

# Double check atse_psi
print("Cells with all zero PSI values:", (atse_psi.sum(axis=1) == 0).sum())
print("Exons with all zero PSI values:", (atse_psi.sum(axis=0) == 0).sum())

Cells with all zero counts: 0
Exons with all zero counts: 0
Cells with all zero PSI values: 0
Exons with all zero PSI values: 0


In [9]:
atse_psi = atse_psi.T
atse_counts = atse_counts.T

In [10]:
# Save as a file
atse_psi.to_csv(os.path.join(sim_dir, test_dir, atse_psi_path), sep="\t", compression="gzip")
atse_counts.to_csv(os.path.join(sim_dir, test_dir, atse_counts_path), sep="\t", compression="gzip")

In [11]:
# Perform PCA
pca = PCA(n_components=10)  # Reduce to 10 dimensions
latent_pca = pca.fit_transform(atse_psi.T)

# Convert to DataFrame
latent_df = pd.DataFrame(latent_pca, index=atse_psi.T.index)
print(latent_df.shape)

# add columns to latent_df PC_{} 
latent_df.columns = ["PC_" + str(c) for c in latent_df.columns]
latent_df.to_csv(os.path.join(sim_dir, test_dir, "latent_space.tab.gz"), sep="\t", compression="gzip")

(18269, 10)


In [12]:
# Now pass the file paths to Psix
psix_object = psix.Psix(
    psi_table=os.path.join(sim_dir, test_dir, atse_psi_path),
    counts_per_event=os.path.join(sim_dir, test_dir, atse_counts_path),
    counts_type="reads")

Transforming read counts into TPM.
Transforming TPM into mRNA counts.


100%|██████████| 18269/18269 [10:17<00:00, 29.58it/s]


In [13]:
# psix.psix.build_lookup(out_dir = 'lookup/') # This function will create a directory named "lookup"

In [14]:
psix_object.adata.uns['psi']

,ATSE_3,ATSE_5,ATSE_15,ATSE_62,ATSE_69,ATSE_70,ATSE_77,ATSE_81,ATSE_84,ATSE_109,...,ATSE_62048,ATSE_62054,ATSE_62070,ATSE_62082,ATSE_62088,ATSE_62090,ATSE_62094,ATSE_62099,ATSE_62114,ATSE_62124
A1_B000826,0.0,0.0,0.0,0.0,0.806122,0.2,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
A1_B001176,0.0,0.0,0.0,0.0,0.812680,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.8,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
A1_B003279,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
A1_B003281,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
A1_B003290,0.0,0.0,0.0,0.0,0.923077,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.571429,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P9_B003914,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
P9_B003921,0.0,0.0,0.0,0.0,0.838685,0.0,0.0,0.0,0.0,0.262295,...,0.0,0.0,0.0,0.0,0.000000,0.755682,0.0,0.0,0.0,0.0
P9_D045315,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
P9_D045318,0.0,0.0,0.0,0.0,0.816794,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0


In [15]:
psix_object.run_psix(lookup = 'lookup/', latent=os.path.join(sim_dir, test_dir, "latent_space.tab.gz"))

100%|██████████| 50/50 [00:00<00:00, 58.62it/s]

Computing cell-cell metric...



100%|██████████| 18269/18269 [00:02<00:00, 7080.96it/s] 

Successfully computed cell-cell metric
Computing Psix score in 4855 exons



100%|██████████| 4855/4855 [03:10<00:00, 25.50it/s]

Successfully computed Psix score of exons.
Estimating p-values. This might take a while...



100%|██████████| 25/25 [23:10<00:00, 55.62s/it]


Successfully estimated p-values
